In [ ]:
!pip install torch torchvision transformers pillow editdistance

In [ ]:
import os
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms

from transformers import TrOCRProcessor, VisionEncoderDecoderModel

!unzip fix_19_38.zip

class OCRDataset(Dataset):
    def __init__(self, root_dir, processor):
        self.root_dir = root_dir
        self.processor = processor
        self.samples = []

        # обходим все страницы
        for page in os.listdir(root_dir):
            page_path = os.path.join(root_dir, page)

            if not os.path.isdir(page_path):
                continue

            text_dir = os.path.join(page_path, "text")

            if not os.path.exists(text_dir):
                continue

            # ищем пары внутри text/
            for file in os.listdir(text_dir):
                if file.endswith(".png"):
                    txt_file = file.replace(".png", ".txt")

                    img_path = os.path.join(text_dir, file)
                    txt_path = os.path.join(text_dir, txt_file)

                    if os.path.exists(txt_path):
                        text = open(txt_path, encoding="utf-8").read().strip()

                        if text:  # фильтр пустых
                            self.samples.append((img_path, txt_path))

        print(f"✅ Найдено {len(self.samples)} примеров")

        self.augment = transforms.Compose([
            transforms.RandomRotation(2),
            transforms.ColorJitter(brightness=0.2, contrast=0.2)
        ])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, txt_path = self.samples[idx]

        image = Image.open(img_path).convert("RGB")
        image = self.augment(image)

        with open(txt_path, encoding="utf-8") as f:
            text = f.read().strip()

        pixel_values = self.processor(images=image, return_tensors="pt").pixel_values.squeeze()
        labels = self.processor.tokenizer(text, return_tensors="pt").input_ids.squeeze()

        return {
            "pixel_values": pixel_values,
            "labels": labels
        }

Archive:  fix_19_38.zip
   creating: fix_19_38/
   creating: fix_19_38/page_020/
   creating: fix_19_38/page_020/formula/
  inflating: fix_19_38/page_020/formula/region_00.png  
 extracting: fix_19_38/page_020/formula/region_01.png  
  inflating: fix_19_38/page_020/formula/region_02.png  
  inflating: fix_19_38/page_020/formula/region_03.png  
  inflating: fix_19_38/page_020/formula/region_04.png  
  inflating: fix_19_38/page_020/formula/region_05.png  
  inflating: fix_19_38/page_020/formula/region_06.png  
  inflating: fix_19_38/page_020/formula/region_07.png  
  inflating: fix_19_38/page_020/formula/region_08.png  
   creating: fix_19_38/page_020/picture/
  inflating: fix_19_38/page_020/picture/region_00.png  
  inflating: fix_19_38/page_020/picture/region_01.png  
  inflating: fix_19_38/page_020/picture/region_02.png  
  inflating: fix_19_38/page_020/picture/region_03.png  
  inflating: fix_19_38/page_020/picture/region_04.png  
  inflating: fix_19_38/page_020/picture/region_05.png

In [ ]:
def collate_fn(batch):
    pixel_values = torch.stack([x["pixel_values"] for x in batch])
    labels = [x["labels"] for x in batch]

    labels = torch.nn.utils.rnn.pad_sequence(
        labels,
        batch_first=True,
        padding_value=-100
    )

    return {
        "pixel_values": pixel_values,
        "labels": labels
    }

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

processor = TrOCRProcessor.from_pretrained("Daniil-Domino/trocr-base-ru-dialectic")
model = VisionEncoderDecoderModel.from_pretrained("Daniil-Domino/trocr-base-ru-dialectic")

# добавим символы гидродинамики
new_tokens = ["ρ", "∂", "∇", "u", "v", "w"]
processor.tokenizer.add_tokens(new_tokens)
model.decoder.resize_token_embeddings(len(processor.tokenizer))

model.to(device)

Device: cuda


preprocessor_config.json:   0%|          | 0.00/364 [00:00<?, ?B/s]

The image processor of type `ViTImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.34G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/480 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/205 [00:00<?, ?B/s]

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


VisionEncoderDecoderModel(
  (encoder): ViTModel(
    (embeddings): ViTEmbeddings(
      (patch_embeddings): ViTPatchEmbeddings(
        (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): ViTEncoder(
      (layer): ModuleList(
        (0-11): 12 x ViTLayer(
          (attention): ViTAttention(
            (attention): ViTSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=False)
              (key): Linear(in_features=768, out_features=768, bias=False)
              (value): Linear(in_features=768, out_features=768, bias=False)
            )
            (output): ViTSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
          )
          (intermediate): ViTIntermediate(
            (dense): Linear(in_features=768, out_features=3072, bias=True)
            (i

In [ ]:
data_dir = "fix_19_38"  # путь к папке с page_XXX

dataset = OCRDataset(data_dir, processor)

print("Размер датасета:", len(dataset))

loader = DataLoader(
    dataset,
    batch_size=4,
    shuffle=True,
    collate_fn=collate_fn
)

print("Samples:", len(dataset))

✅ Найдено 264 примеров
Размер датасета: 264
Samples: 264


In [ ]:
processor = TrOCRProcessor.from_pretrained("Daniil-Domino/trocr-base-ru-dialectic")
model = VisionEncoderDecoderModel.from_pretrained("Daniil-Domino/trocr-base-ru-dialectic")

# спец символы
new_tokens = ["ρ", "∂", "∇", "u", "v", "w"]
processor.tokenizer.add_tokens(new_tokens)
model.decoder.resize_token_embeddings(len(processor.tokenizer))

# 🔥 ВАЖНО (фикс ошибки)
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.eos_token_id = processor.tokenizer.sep_token_id

model.to(device)

Loading weights:   0%|          | 0/480 [00:00<?, ?it/s]

VisionEncoderDecoderModel(
  (encoder): ViTModel(
    (embeddings): ViTEmbeddings(
      (patch_embeddings): ViTPatchEmbeddings(
        (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): ViTEncoder(
      (layer): ModuleList(
        (0-11): 12 x ViTLayer(
          (attention): ViTAttention(
            (attention): ViTSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=False)
              (key): Linear(in_features=768, out_features=768, bias=False)
              (value): Linear(in_features=768, out_features=768, bias=False)
            )
            (output): ViTSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
          )
          (intermediate): ViTIntermediate(
            (dense): Linear(in_features=768, out_features=3072, bias=True)
            (i

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)

epochs = 20

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for batch in loader:
        pixel_values = batch["pixel_values"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(pixel_values=pixel_values, labels=labels)
        loss = outputs.loss

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}: loss = {total_loss / len(loader):.4f}")

Epoch 1: loss = 0.5893
Epoch 2: loss = 0.3843
Epoch 3: loss = 0.2729
Epoch 4: loss = 0.1922
Epoch 5: loss = 0.1357
Epoch 6: loss = 0.0916
Epoch 7: loss = 0.0711
Epoch 8: loss = 0.0447
Epoch 9: loss = 0.0352
Epoch 10: loss = 0.0297
Epoch 11: loss = 0.0243
Epoch 12: loss = 0.0229
Epoch 13: loss = 0.0220
Epoch 14: loss = 0.0164
Epoch 15: loss = 0.0149
Epoch 16: loss = 0.0150
Epoch 17: loss = 0.0160
Epoch 18: loss = 0.0130
Epoch 19: loss = 0.0126
Epoch 20: loss = 0.0112


In [ ]:
model.save_pretrained("model")
processor.save_pretrained("model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

['model/processor_config.json']

In [ ]:
def predict(image_path):
    model.eval()

    image = Image.open(image_path).convert("RGB")
    pixel_values = processor(images=image, return_tensors="pt").pixel_values.to(device)

    with torch.no_grad():
        generated_ids = model.generate(pixel_values)

    text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return text

In [ ]:
print(predict("fix_19_38/page_023/text/region_06.png"))

Квазиодномерное приближ.:пов-ти пост. пар-мов замен. на пл-
